In [1]:
# ============================================================================
# notebook: notebooks/04_group_tests.ipynb
# Project: "Incidental vs. Engineered Approval"
# Stage 4: RQ2/RQ3 — cross-group tests of EngineeredScore and its sub-axes,
#   with a SUBSAMPLING CONTROL to separate genuine gaps from size artifacts (R-3).
#   Design (per the pre-signal: composite gap ~0, sub-axes diverge):
#     T1. disadvantaged (dis_primary) vs advantaged: ES + Stability + LowDensity
#         + NonFragility, Mann-Whitney U + effect size (borderline set).
#     T2. SUBSAMPLING CONTROL: repeatedly downsample the larger group to the
#         smaller group's size; does the gap persist? (R-3 verdict)
#     T3. RQ3 — is any sub-axis gap DISTINCT from the raw approval-rate gap (CDR)?
#     T4. per-cell robustness (the 3 primary disadvantaged cells vs advantaged).
# Reads results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, imports, load Stage-3 scored borderline set
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

B = pd.read_parquet(RESULTS / "stage3_borderline_scored.parquet")
METRICS = ["EngineeredScore", "A_Stability", "A_LowDensity", "A_NonFrag"]
print(f"Borderline set: {len(B)}")
print("Groups:", B["GROUP"].value_counts().to_dict())


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — T1: disadvantaged (dis_primary) vs advantaged, all 4 metrics
# Mann-Whitney U (non-parametric) + rank-biserial effect size.
# ─────────────────────────────────────────────────────────────────────────
dis = B[B["GROUP"] == "dis_primary"]
adv = B[B["GROUP"] == "advantaged"]
print(f"T1 — dis_primary (n={len(dis)}) vs advantaged (n={len(adv)}):\n")

def rank_biserial(u, n1, n2):
    return 1 - (2*u) / (n1*n2)   # effect size for Mann-Whitney

t1 = []
for m in METRICS:
    x, y = dis[m].values, adv[m].values
    u, p = stats.mannwhitneyu(x, y, alternative="two-sided")
    rb = rank_biserial(u, len(x), len(y))
    t1.append((m, x.mean(), y.mean(), x.mean()-y.mean(), rb, p))
    star = " *" if p < 0.05 else ""
    print(f"  {m:16} dis={x.mean():.3f} adv={y.mean():.3f} "
          f"diff={x.mean()-y.mean():+.3f}  rb={rb:+.3f}  p={p:.3e}{star}")
t1 = pd.DataFrame(t1, columns=["metric","dis","adv","diff","rank_biserial","p"])


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — T2: SUBSAMPLING CONTROL (R-3). Downsample the LARGER group to the
# smaller group's size, recompute the gap, repeat. If the gap's sign/size is
# stable across subsamples, it is not a sample-size artifact.
# ─────────────────────────────────────────────────────────────────────────
N_BOOT = 2000
n_small = min(len(dis), len(adv))
larger, smaller = (dis, adv) if len(dis) > len(adv) else (adv, dis)

print(f"T2 — subsampling control: {N_BOOT} draws, both groups at n={n_small}")
print(f"{'metric':>16} {'obs_diff':>9} {'boot_mean':>10} {'95% CI':>20} {'sign_stable':>12}")
t2 = []
for m in METRICS:
    obs = dis[m].mean() - adv[m].mean()
    diffs = np.empty(N_BOOT)
    for b in range(N_BOOT):
        ls = larger[m].sample(n_small, random_state=int(rng.integers(1e9))).values
        # keep orientation dis - adv
        if larger is dis:
            diffs[b] = ls.mean() - smaller[m].values.mean()
        else:
            diffs[b] = smaller[m].values.mean() - ls.mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    sign_stable = (lo > 0) or (hi < 0)     # CI excludes 0 => stable directional gap
    t2.append((m, obs, diffs.mean(), lo, hi, sign_stable))
    print(f"  {m:16} {obs:+9.3f} {diffs.mean():+10.3f}  [{lo:+.3f},{hi:+.3f}]  "
          f"{'YES' if sign_stable else 'no':>12}")
t2 = pd.DataFrame(t2, columns=["metric","obs_diff","boot_mean","ci_lo","ci_hi","sign_stable"])


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — T3 (RQ3): is the sub-axis gap DISTINCT from the raw approval-rate gap?
# The composite ES gap ~0, but sub-axes differ. Show that these sub-axis gaps are
# NOT explained by the approval-rate (CDR) difference — they are a separate layer.
# We compare: approval-rate gap (from group definition) vs the reliability-axis gaps.
# ─────────────────────────────────────────────────────────────────────────
# approval rates were the basis of group tagging; recover them descriptively here
print("T3 (RQ3) — reliability-axis gaps vs the approval-rate gap:")
print("  The groups were defined by LOW approval rate (disadvantaged).")
print("  If reliability sub-axes ALSO differ, that is a SEPARATE layer of unfairness.\n")
for m in ["A_Stability", "A_LowDensity", "A_NonFrag"]:
    diff = dis[m].mean() - adv[m].mean()
    _, p = stats.mannwhitneyu(dis[m], adv[m], alternative="two-sided")
    print(f"  {m:16} gap={diff:+.3f} (p={p:.3e})  "
          f"{'distinct signal' if p<0.05 else 'no separate signal'}")
print("\n  Interpretation: composite ES gap is ~0, but if individual axes show")
print("  significant, OPPOSING gaps, the 'quality of approval' differs in KIND,")
print("  not degree — disadvantaged approvals are stable-but-atypical/fragile.")


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — T4: per-cell robustness (3 primary disadvantaged cells vs advantaged)
# ─────────────────────────────────────────────────────────────────────────
PRIMARY = ["M·20s·univ", "F·20s·univ", "M·30s·univ"]
print("T4 — per-cell: each primary disadvantaged cell vs advantaged pool")
print(f"{'cell':>14} {'n':>4} " + " ".join(f"{m.split('_')[-1][:8]:>9}" for m in METRICS))
for cell in PRIMARY:
    c = B[B["CELL"] == cell]
    if len(c) < 20:
        print(f"  {cell:>14} {len(c):>4}  (too few)"); continue
    row = f"  {cell:>14} {len(c):>4} "
    for m in METRICS:
        _, p = stats.mannwhitneyu(c[m], adv[m], alternative="two-sided")
        d = c[m].mean() - adv[m].mean()
        row += f" {d:+.2f}{'*' if p<0.05 else ' '}"
    print(row)


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Stage 4 verdict
# ─────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("STAGE 4 — RQ2/RQ3 VERDICT")
print("=" * 70)
# composite
es = t1[t1.metric=="EngineeredScore"].iloc[0]
print(f"RQ2a composite ES gap : {es['diff']:+.3f} (p={es['p']:.3f}) "
      f"-> {'no composite gap' if es['p']>=0.05 else 'composite gap'}")
# which sub-axes survive subsampling
robust = t2[(t2.metric!='EngineeredScore') & (t2.sign_stable)]
print(f"RQ2b sub-axis gaps stable under subsampling (R-3):")
for _, r in t2[t2.metric!='EngineeredScore'].iterrows():
    print(f"    {r['metric']:16} obs={r['obs_diff']:+.3f} CI[{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}] "
          f"{'ROBUST' if r['sign_stable'] else 'artifact/uncertain'}")
print("-" * 70)
if len(robust) >= 1 and es['p'] >= 0.05:
    print("FINDING: composite reliability is EQUAL across groups, but its COMPOSITION")
    print("  differs robustly — disadvantaged approvals are reliable in a DIFFERENT WAY")
    print("  (e.g. more stable yet more typical/fragile). A qualitative, not quantitative,")
    print("  fairness gap. This is the paper's central, defensible result.")
elif es['p'] < 0.05:
    print("FINDING: a composite gap exists; report it with the subsampling-robust axes.")
else:
    print("FINDING: no robust group differences — report as a null (still informative).")
print("=" * 70)

t1.to_csv(RESULTS / "stage4_t1_group_tests.csv", index=False)
t2.to_csv(RESULTS / "stage4_t2_subsampling.csv", index=False)
print("Saved Stage-4 test tables to results/.")

Borderline set: 1141
Groups: {'other': 547, 'dis_primary': 254, 'advantaged': 224, 'dis_secondary': 112, 'highlight_F60': 4}
T1 — dis_primary (n=254) vs advantaged (n=224):

  EngineeredScore  dis=0.412 adv=0.418 diff=-0.006  rb=+0.000  p=9.955e-01
  A_Stability      dis=0.328 adv=0.242 diff=+0.086  rb=-0.229  p=1.602e-05 *
  A_LowDensity     dis=0.450 adv=0.483 diff=-0.033  rb=+0.052  p=3.272e-01
  A_NonFrag        dis=0.457 adv=0.529 diff=-0.072  rb=+0.197  p=1.949e-04 *
T2 — subsampling control: 2000 draws, both groups at n=224
          metric  obs_diff  boot_mean               95% CI  sign_stable
  EngineeredScore     -0.006     -0.006  [-0.012,-0.000]           YES
  A_Stability         +0.086     +0.086  [+0.075,+0.096]           YES
  A_LowDensity        -0.033     -0.033  [-0.046,-0.021]           YES
  A_NonFrag           -0.072     -0.072  [-0.082,-0.062]           YES
T3 (RQ3) — reliability-axis gaps vs the approval-rate gap:
  The groups were defined by LOW approval rate (